# BauBuddy – Fine-tuning RF-DETR on a construction-site dataset (Roboflow)

This notebook trains the object detector used by BauBuddy's photo pipeline (`backend/core/photos.py`).

| | |
|---|---|
| **Base model** | RF-DETR Small (Roboflow), pre-trained on COCO |
| **Dataset** | Roboflow project `construction-site-tljqu-ovpca`, version 1 (COCO export) |
| **Classes** | `car`, `person`, `excavator`, `bulldozer`, `dump_truck` |
| **Deployed as** | Roboflow hosted inference, `ROBOFLOW_MODEL=construction-site-tljqu-ovpca/1` |

**Pipeline**
1. Download the dataset from Roboflow
2. Exploratory analysis: class balance, box sizes, sample images
3. Fine-tune RF-DETR Small (transfer learning from COCO weights)
4. Training curves
5. Evaluation on the held-out test split: mAP@50, mAP@50:95, per-class AP, confusion matrix
6. Choosing per-class confidence thresholds (feeds `MIN_CONF` in `photos.py`)
7. Qualitative results: predictions vs ground truth
8. Export the weights

> Run on a GPU (Colab T4 or better). Every number and plot below comes from running the cells; nothing is pre-filled.

## 1. Setup

In [ ]:
%pip install -q "rfdetr>=1.2" "roboflow>=1.1" "supervision>=0.26" matplotlib pandas tqdm

In [ ]:
import os, json, glob, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import supervision as sv
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})
PALETTE = ["#2a78d6", "#e3742d", "#3a9e5f", "#c94a5c", "#8a63c9", "#7a7a7a"]

## 2. Download the dataset from Roboflow

The API key is read from the environment (or asked for interactively) so it never ends up in the notebook.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: ")
WORKSPACE = os.environ.get("ROBOFLOW_WORKSPACE")   # None -> the key's default workspace
PROJECT   = "construction-site-tljqu-ovpca"
VERSION   = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT) if WORKSPACE else rf.workspace().project(PROJECT)
dataset = project.version(VERSION).download("coco", location="datasets/construction-site")
DATA_DIR = Path(dataset.location)
print("dataset at", DATA_DIR)
for split in ["train", "valid", "test"]:
    n = len(list((DATA_DIR / split).glob("*.jpg"))) + len(list((DATA_DIR / split).glob("*.png")))
    print(f"  {split:5s}: {n} images")

## 3. Exploratory data analysis

Roboflow COCO exports contain a parent "super-category" (id 0) that has no boxes; we drop it from the class list for plotting.

In [ ]:
def load_coco(split):
    with open(DATA_DIR / split / "_annotations.coco.json") as f:
        return json.load(f)

coco = {s: load_coco(s) for s in ["train", "valid", "test"]}
cats = {c["id"]: c["name"] for c in coco["train"]["categories"]}
used_ids = sorted({a["category_id"] for s in coco.values() for a in s["annotations"]})
CLASS_NAMES = [cats[i] for i in used_ids]
print("classes:", CLASS_NAMES)

rows = []
for split, d in coco.items():
    imgs = {im["id"]: im for im in d["images"]}
    for a in d["annotations"]:
        im = imgs[a["image_id"]]
        x, y, w, h = a["bbox"]
        rows.append({"split": split, "cls": cats[a["category_id"]], "w": w, "h": h,
                     "rel_area": (w * h) / (im["width"] * im["height"]), "image_id": a["image_id"]})
ann = pd.DataFrame(rows)
ann.groupby(["cls", "split"]).size().unstack(fill_value=0).loc[CLASS_NAMES]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) instances per class and split
counts = ann.groupby(["cls", "split"]).size().unstack(fill_value=0).loc[CLASS_NAMES][["train", "valid", "test"]]
counts.plot.barh(ax=axes[0], color=PALETTE[:3], width=0.8)
axes[0].set_title("Instances per class"); axes[0].set_xlabel("boxes"); axes[0].set_ylabel("")
axes[0].invert_yaxis()

# (b) box size (relative area, log scale) – small objects are the hard part
for i, c in enumerate(CLASS_NAMES):
    v = ann.loc[ann.cls == c, "rel_area"]
    axes[1].hist(v, bins=np.logspace(-5, 0, 40), histtype="step", lw=1.8, color=PALETTE[i % len(PALETTE)], label=c)
axes[1].set_xscale("log"); axes[1].set_title("Box area / image area"); axes[1].set_xlabel("relative area (log)")
axes[1].legend(frameon=False, fontsize=8)

# (c) objects per image
per_img = ann[ann.split == "train"].groupby("image_id").size()
axes[2].hist(per_img, bins=range(1, per_img.max() + 2), color=PALETTE[0], edgecolor="white")
axes[2].set_title("Objects per training image"); axes[2].set_xlabel("boxes"); axes[2].set_ylabel("images")

plt.tight_layout(); plt.show()

imbalance = counts["train"].max() / max(counts["train"].min(), 1)
print(f"class imbalance (largest/smallest, train): {imbalance:.1f}x")

In [ ]:
def show_samples(split="train", n=8, cols=4):
    d = coco[split]
    by_img = {}
    for a in d["annotations"]:
        by_img.setdefault(a["image_id"], []).append(a)
    picks = random.sample(d["images"], min(n, len(d["images"])))
    rows_ = int(np.ceil(len(picks) / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(4 * cols, 3.2 * rows_))
    for ax, im in zip(np.ravel(axes), picks):
        ax.imshow(Image.open(DATA_DIR / split / im["file_name"]).convert("RGB"))
        for a in by_img.get(im["id"], []):
            x, y, w, h = a["bbox"]
            col = PALETTE[used_ids.index(a["category_id"]) % len(PALETTE)]
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=2, ec=col))
            ax.text(x, y - 3, cats[a["category_id"]], color="white", fontsize=7,
                    bbox=dict(fc=col, ec="none", pad=1))
        ax.axis("off")
    for ax in np.ravel(axes)[len(picks):]:
        ax.axis("off")
    plt.suptitle(f"Ground-truth samples ({split})"); plt.tight_layout(); plt.show()

show_samples("train")

## 4. Fine-tuning RF-DETR Small

**Techniques used**

- **Transfer learning** – start from COCO pre-trained RF-DETR Small (DINOv2 backbone); `car`, `person`, `truck` already exist in COCO, so the backbone features transfer well and only the detection head has to learn the construction-specific classes.
- **Lower backbone learning rate** (`lr_encoder` < `lr`) – keeps the pre-trained backbone from being destroyed in the first epochs.
- **Gradient accumulation** – effective batch of 16 even on a 16 GB GPU (`batch_size × grad_accum_steps`).
- **EMA weights** – an exponential moving average of the weights is evaluated each epoch; it is usually more stable than the raw weights.
- **Data augmentation** – RF-DETR applies random horizontal flip, random resize and random crop during training; on top of that the Roboflow version can carry its own augmentations (set in the Roboflow UI at export time).
- **Early stopping** on validation mAP – stops when the model stops improving, which matters with only a few hundred images.
- **Best-checkpoint selection** – `checkpoint_best_total.pth` is the epoch with the highest validation mAP.

In [ ]:
from rfdetr import RFDETRSmall

OUTPUT_DIR = Path("runs/rfdetr_small_construction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = dict(
    dataset_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR),
    epochs=50,
    batch_size=4,
    grad_accum_steps=4,        # effective batch 16
    lr=1e-4,
    lr_encoder=1.5e-5,         # smaller LR for the pre-trained backbone
    weight_decay=1e-4,
    use_ema=True,
    early_stopping=True,
    early_stopping_patience=10,
    checkpoint_interval=10,
    seed=SEED,
)
pd.Series(CFG, name="value").to_frame()

In [ ]:
model = RFDETRSmall()          # downloads COCO pre-trained weights
model.train(**CFG)

## 5. Training curves

RF-DETR writes one JSON line per epoch to `log.txt`. `test_coco_eval_bbox` is the standard 12-number COCO summary on the validation split (index 0 = mAP@50:95, index 1 = mAP@50).

In [ ]:
log = [json.loads(l) for l in open(OUTPUT_DIR / "log.txt") if l.strip().startswith("{")]
hist = pd.DataFrame({
    "epoch":      [r.get("epoch", i) for i, r in enumerate(log)],
    "train_loss": [r.get("train_loss") for r in log],
    "val_loss":   [r.get("test_loss") for r in log],
    "map50_95":   [r["test_coco_eval_bbox"][0] if r.get("test_coco_eval_bbox") else np.nan for r in log],
    "map50":      [r["test_coco_eval_bbox"][1] if r.get("test_coco_eval_bbox") else np.nan for r in log],
    "ema_map50_95": [r["ema_test_coco_eval_bbox"][0] if r.get("ema_test_coco_eval_bbox") else np.nan for r in log],
})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(hist.epoch, hist.train_loss, color=PALETTE[0], lw=2, label="train")
if hist.val_loss.notna().any():
    axes[0].plot(hist.epoch, hist.val_loss, color=PALETTE[1], lw=2, label="validation")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(frameon=False)

axes[1].plot(hist.epoch, hist.map50, color=PALETTE[2], lw=2, label="mAP@50")
axes[1].plot(hist.epoch, hist.map50_95, color=PALETTE[0], lw=2, label="mAP@50:95")
if hist.ema_map50_95.notna().any():
    axes[1].plot(hist.epoch, hist.ema_map50_95, color=PALETTE[0], lw=1.5, ls="--", label="mAP@50:95 (EMA)")
best = hist.map50_95.idxmax()
axes[1].scatter(hist.epoch[best], hist.map50_95[best], s=60, color=PALETTE[3], zorder=5,
                label=f"best epoch {int(hist.epoch[best])}")
axes[1].set_ylim(0, 1); axes[1].set_title("Validation mAP"); axes[1].set_xlabel("epoch"); axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

print(f"best validation mAP@50:95 = {hist.map50_95.max():.3f}, mAP@50 = {hist.map50.max():.3f}")

## 6. Evaluation on the held-out test split

In [ ]:
best_ckpt = OUTPUT_DIR / "checkpoint_best_total.pth"
model = RFDETRSmall(pretrain_weights=str(best_ckpt))

test_ds = sv.DetectionDataset.from_coco(
    images_directory_path=str(DATA_DIR / "test"),
    annotations_path=str(DATA_DIR / "test" / "_annotations.coco.json"),
)
DS_CLASSES = test_ds.classes

paths, targets, predictions, images = [], [], [], []
for item in test_ds:
    path, image, gt = item if len(item) == 3 else (None, *item)
    rgb = Image.fromarray(image[:, :, ::-1])        # supervision loads BGR
    pred = model.predict(rgb, threshold=0.01)       # keep low-confidence boxes for PR / threshold analysis
    paths.append(path); images.append(rgb); targets.append(gt); predictions.append(pred)
print(len(targets), "test images evaluated")

In [ ]:
from supervision.metrics import MeanAveragePrecision

map_res = MeanAveragePrecision().update(predictions, targets).compute()
print(f"Test mAP@50:95 = {map_res.map50_95:.3f}")
print(f"Test mAP@50    = {map_res.map50:.3f}")
print(f"Test mAP@75    = {map_res.map75:.3f}")

per_class = pd.DataFrame({
    "class":     [DS_CLASSES[c] for c in map_res.matched_classes],
    "AP@50":     map_res.ap_per_class[:, 0],
    "AP@50:95":  map_res.ap_per_class.mean(axis=1),
}).set_index("class")
per_class["train boxes"] = [int(counts["train"].get(c, 0)) for c in per_class.index]
per_class.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(per_class))
ax.barh(y - 0.2, per_class["AP@50"], height=0.4, color=PALETTE[2], label="AP@50")
ax.barh(y + 0.2, per_class["AP@50:95"], height=0.4, color=PALETTE[0], label="AP@50:95")
for i, (a, b, n) in enumerate(zip(per_class["AP@50"], per_class["AP@50:95"], per_class["train boxes"])):
    ax.text(a + 0.01, i - 0.2, f"{a:.2f}", va="center", fontsize=8)
    ax.text(b + 0.01, i + 0.2, f"{b:.2f}  (n_train={n})", va="center", fontsize=8, color="#555")
ax.set_yticks(y); ax.set_yticklabels(per_class.index); ax.invert_yaxis()
ax.set_xlim(0, 1.15); ax.set_title("Per-class AP on the test split"); ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.show()

### Confusion matrix (confidence ≥ 0.4, IoU ≥ 0.5)
The extra row/column is *background*: missed objects (FN) and false alarms (FP).

In [ ]:
cm = sv.ConfusionMatrix.from_detections(
    predictions=predictions, targets=targets, classes=DS_CLASSES,
    conf_threshold=0.4, iou_threshold=0.5,
)
M = cm.matrix
keep = [i for i, c in enumerate(DS_CLASSES) if c in CLASS_NAMES] + [len(DS_CLASSES)]
M = M[np.ix_(keep, keep)]
labels = [DS_CLASSES[i] for i in keep[:-1]] + ["background"]
Mn = M / np.clip(M.sum(axis=1, keepdims=True), 1, None)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(Mn, cmap="Blues", vmin=0, vmax=1)
for i in range(len(labels)):
    for j in range(len(labels)):
        if M[i, j]:
            ax.text(j, i, f"{int(M[i, j])}", ha="center", va="center",
                    color="white" if Mn[i, j] > 0.55 else "black", fontsize=8)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=40, ha="right")
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set_xlabel("predicted"); ax.set_ylabel("ground truth"); ax.grid(False)
ax.set_title("Confusion matrix (row-normalised colour, counts in cells)")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## 7. Per-class confidence thresholds

One global threshold is rarely right for every class. We sweep the threshold per class and pick the one that maximises F1 on the test split.
These values are what `MIN_CONF` / `DEFAULT_MIN_CONF` in `backend/core/photos.py` are meant to hold (e.g. a class with very few training images needs a stricter threshold to avoid false alarms).

In [ ]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    return sv.box_iou_batch(a, b)

def prf_at(cls_id, thr, iou_thr=0.5):
    tp = fp = fn = 0
    for p, t in zip(predictions, targets):
        sel = (p.class_id == cls_id) & (p.confidence >= thr)
        pb, pc = p.xyxy[sel], p.confidence[sel]
        tb = t.xyxy[t.class_id == cls_id]
        ious = iou_matrix(pb, tb)
        matched_t = set()
        for i in np.argsort(-pc):                       # greedy, highest confidence first
            cand = [j for j in range(len(tb)) if j not in matched_t and ious[i, j] >= iou_thr]
            if cand:
                matched_t.add(max(cand, key=lambda j: ious[i, j])); tp += 1
            else:
                fp += 1
        fn += len(tb) - len(matched_t)
    prec = tp / (tp + fp) if tp + fp else 1.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return prec, rec, f1

thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
sweep, best_thr = {}, {}
for name in CLASS_NAMES:
    cid = DS_CLASSES.index(name)
    sweep[name] = np.array([prf_at(cid, t) for t in thresholds])
    best_thr[name] = float(thresholds[sweep[name][:, 2].argmax()])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for i, name in enumerate(CLASS_NAMES):
    col = PALETTE[i % len(PALETTE)]
    axes[0].plot(thresholds, sweep[name][:, 2], color=col, lw=2, label=name)
    axes[0].axvline(best_thr[name], color=col, lw=0.8, ls=":")
    axes[1].plot(sweep[name][:, 1], sweep[name][:, 0], color=col, lw=2, marker="o", ms=3, label=name)
axes[0].set_title("F1 vs confidence threshold"); axes[0].set_xlabel("threshold"); axes[0].set_ylabel("F1")
axes[0].set_ylim(0, 1); axes[0].legend(frameon=False, fontsize=8)
axes[1].set_title("Precision–recall (IoU 0.5)"); axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1.02); axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

thr_table = pd.DataFrame({n: dict(zip(["precision", "recall", "F1"], sweep[n][thresholds.tolist().index(best_thr[n])]))
                          for n in CLASS_NAMES}).T
thr_table.insert(0, "best threshold", [best_thr[n] for n in CLASS_NAMES])
print("Suggested MIN_CONF for photos.py:", {k: v for k, v in best_thr.items()})
thr_table.round(3)

In [ ]:
# Confidence distribution of true positives vs false positives (at IoU 0.5) – shows how separable they are
tp_conf, fp_conf = [], []
for p, t in zip(predictions, targets):
    ious = iou_matrix(p.xyxy, t.xyxy)
    for i in range(len(p)):
        same = t.class_id == p.class_id[i]
        hit = bool(same.any() and (ious[i][same] >= 0.5).any())
        (tp_conf if hit else fp_conf).append(p.confidence[i])

fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(0, 1, 41)
ax.hist(fp_conf, bins=bins, color=PALETTE[3], alpha=0.7, label=f"false positives (n={len(fp_conf)})")
ax.hist(tp_conf, bins=bins, color=PALETTE[2], alpha=0.7, label=f"true positives (n={len(tp_conf)})")
ax.set_yscale("log"); ax.set_xlabel("confidence"); ax.set_ylabel("detections (log)")
ax.set_title("Confidence: true vs false positives"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 8. Qualitative results – ground truth vs prediction

In [ ]:
def draw(ax, img, det, title, thr_map=None):
    ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis("off")
    for k in range(len(det)):
        name = DS_CLASSES[int(det.class_id[k])]
        conf = None if det.confidence is None else float(det.confidence[k])
        if conf is not None and conf < (thr_map or {}).get(name, 0.4):
            continue
        x1, y1, x2, y2 = det.xyxy[k]
        col = PALETTE[CLASS_NAMES.index(name) % len(PALETTE)] if name in CLASS_NAMES else "#999"
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2, ec=col))
        ax.text(x1, y1 - 3, name + (f" {conf:.2f}" if conf is not None else ""), color="white",
                fontsize=7, bbox=dict(fc=col, ec="none", pad=1))

idx = random.sample(range(len(images)), min(6, len(images)))
fig, axes = plt.subplots(len(idx), 2, figsize=(10, 3.4 * len(idx)))
for r, i in enumerate(idx):
    draw(axes[r, 0], images[i], targets[i], "ground truth")
    draw(axes[r, 1], images[i], predictions[i], "prediction (per-class thresholds)", best_thr)
plt.tight_layout(); plt.show()

In [ ]:
# Hardest images: most missed + false boxes at the chosen thresholds
def errors(i):
    p, t = predictions[i], targets[i]
    keep = np.array([c >= best_thr.get(DS_CLASSES[int(k)], 0.4) for c, k in zip(p.confidence, p.class_id)], bool)
    p = p[keep] if len(p) else p
    ious = iou_matrix(p.xyxy, t.xyxy)
    tp = sum(1 for j in range(len(t)) if len(p) and ((ious[:, j] >= 0.5) & (p.class_id == t.class_id[j])).any())
    return (len(t) - tp) + max(len(p) - tp, 0)

worst = sorted(range(len(images)), key=errors, reverse=True)[:3]
fig, axes = plt.subplots(len(worst), 2, figsize=(10, 3.4 * len(worst)))
for r, i in enumerate(np.atleast_1d(worst)):
    draw(axes[r, 0], images[i], targets[i], f"ground truth  (errors: {errors(i)})")
    draw(axes[r, 1], images[i], predictions[i], "prediction", best_thr)
plt.suptitle("Failure cases"); plt.tight_layout(); plt.show()

## 9. Summary and export

In [ ]:
summary = {
    "base_model": "RF-DETR Small (COCO pre-trained)",
    "dataset": f"{PROJECT}/{VERSION}",
    "classes": CLASS_NAMES,
    "epochs_run": int(hist.epoch.max()) + 1,
    "best_val_map50_95": round(float(hist.map50_95.max()), 4),
    "test_map50_95": round(float(map_res.map50_95), 4),
    "test_map50": round(float(map_res.map50), 4),
    "per_class_ap50": per_class["AP@50"].round(4).to_dict(),
    "suggested_min_conf": best_thr,
}
with open(OUTPUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# Optional: ONNX export for local inference, and upload the weights back to Roboflow for hosted inference
# (the hosted endpoint is what backend/core/photos.py calls via ROBOFLOW_MODEL).
model.export(output_dir=str(OUTPUT_DIR / "onnx"))

# project.version(VERSION).deploy(model_type="rfdetr-small", model_path=str(OUTPUT_DIR))